In [1]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import lightning.pytorch as pl
from sklearn.preprocessing import PowerTransformer

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Training on device {device}\n')

Training on device cuda



In [3]:
def get_lat_weights(latitude_values):
    lat_rad = np.deg2rad(latitude_values)
    weights = np.cos(lat_rad)
    return weights / np.mean(weights)

In [4]:
class Normalizer:
    def __init__(self):
        self.mean_in, self.std_in = None, None
        self.mean_out, self.std_out = None, None

    def set_input_statistics(self, mean, std):
        self.mean_in = mean
        self.std_in = std

    def set_output_statistics(self, mean, std):
        self.mean_out = mean
        self.std_out = std

    def normalize(self, data, data_type):
        if data_type == "input":
            return (data - self.mean_in) / self.std_in
        elif data_type == "output":
            return (data - self.mean_out) / self.std_out

    def inverse_transform_output(self, data):
        return data * self.std_out + self.mean_out

    def inverse_transform_output_1dim(self, data, dim):
        return data * self.std_out[:, dim]  + self.mean_out[:, dim]

    def get_mean_std(self):
        return self.std_out, self.mean_out

In [5]:
class ClimateDataset(Dataset):
    def __init__(self, inputs_dask, outputs_dask, output_is_normalized=True):
        self.size = inputs_dask.shape[0]
        print(f"Creating dataset with {self.size} samples...")

        inputs_np = inputs_dask.compute()
        outputs_np = outputs_dask.compute()

        self.inputs = torch.from_numpy(inputs_np).float()
        self.outputs = torch.from_numpy(outputs_np).float()

        if torch.isnan(self.inputs).any() or torch.isnan(self.outputs).any():
            raise ValueError("NaNs found in dataset")

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.inputs[idx], self.outputs[idx]

In [6]:
path = "../processed_data_cse151b_v2_corrupted_ssp245/processed_data_cse151b_v2_corrupted_ssp245.zarr"
input_vars = ["CO2", "SO2", "CH4", "BC", "rsdt"]
output_vars = ["tas", "pr"]
train_ssps = ["ssp126", "ssp370", "ssp585"]
test_ssp = "ssp245"
target_member_id = 0
val_split = 0.1
test_months = 360
batch_size = 32
num_workers = 0
seed = 42
normalizer = Normalizer()

In [7]:
ds = xr.open_zarr(path, consolidated=False, chunks={"time": 24})
spatial_template = ds["rsdt"].isel(time=0, ssp=0, drop=True)
lat = spatial_template.y.values
lon = spatial_template.x.values
area_weights = xr.DataArray(get_lat_weights(lat), dims=["y"], coords={"y": lat})

ds

<xarray.Dataset> Size: 621MB
Dimensions:    (member_id: 3, x: 72, y: 48, longitude: 72, ssp: 4, time: 1021,
                latitude: 48)
Coordinates:
  * member_id  (member_id) int64 24B 0 1 2
    lat        (x, y) float64 28kB dask.array<chunksize=(72, 48), meta=np.ndarray>
  * y          (y) float64 384B -88.59 -84.82 -81.05 ... 81.05 84.82 88.59
  * longitude  (longitude) float64 576B 1.25 6.25 11.25 ... 346.2 351.2 356.2
    lon        (x, y) float64 28kB dask.array<chunksize=(72, 48), meta=np.ndarray>
  * latitude   (latitude) float64 384B -89.05 -85.26 -81.47 ... 85.26 89.05
  * time       (time) object 8kB 2015-01-15 00:00:00 ... 2100-01-15 00:00:00
  * x          (x) float64 576B 1.875 6.875 11.88 16.88 ... 346.9 351.9 356.9
  * ssp        (ssp) <U6 96B 'ssp126' 'ssp245' 'ssp370' 'ssp585'
Data variables:
    pr         (ssp, time, member_id, y, x) float32 169MB dask.array<chunksize=(1, 24, 1, 48, 72), meta=np.ndarray>
    tas        (ssp, time, member_id, y, x) float32 169MB dask.array<chunksize=(1, 24, 1, 48, 72), meta=np.ndarray>
    CH4        (ssp, time) float64 33kB dask.array<chunksize=(1, 24), meta=np.ndarray>
    CO2        (ssp, time) float64 33kB dask.array<chunksize=(1, 24), meta=np.ndarray>
    BC         (ssp, time, latitude, longitude) float64 113MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
    SO2        (ssp, time, latitude, longitude) float64 113MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
    rsdt       (ssp, time, y, x) float32 56MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
Attributes:
    original_member_ids:  ['r10i1p1f1', 'r11i1p1f1', 'r4i1p1f1']
    precipitation_units:  mm/day (converted from kg m-2 s-1)
    source:               CMIP6 data processed for CSE151B
    ssp:                  ssp126

In [8]:
def load_ssp(ssp):
    input_dask, output_dask = [], []
    for var in input_vars:
        da_var = ds[var].sel(ssp=ssp)
        if "latitude" in da_var.dims:
            da_var = da_var.rename({"latitude": "y", "longitude": "x"})
        if "member_id" in da_var.dims:
            da_var = da_var.sel(member_id=target_member_id)
        if set(da_var.dims) == {"time"}:
            da_var = da_var.broadcast_like(spatial_template).transpose("time", "y", "x")
        input_dask.append(da_var.data)

    for var in output_vars:
        da_out = ds[var].sel(ssp=ssp, member_id=target_member_id)
        if "latitude" in da_out.dims:
            da_out = da_out.rename({"latitude": "y", "longitude": "x"})
        output_dask.append(da_out.data)

    return da.stack(input_dask, axis=1), da.stack(output_dask, axis=1)

In [9]:
raw_train_inputs = []
raw_train_outputs = []

ssp_dc = {}

for ssp in train_ssps:
    x, y = load_ssp(ssp)

    if ssp == "ssp370":
        ssp_dc[ssp] = (x[:-test_months], y[:-test_months])
        raw_val_inputs = x[-test_months:]
        raw_val_outputs = y[-test_months:]
    else:
        ssp_dc[ssp] = (x, y)

    raw_train_inputs.append(ssp_dc[ssp][0])
    raw_train_outputs.append(ssp_dc[ssp][1])

raw_train_inputs = da.concatenate(raw_train_inputs, axis=0)
raw_train_outputs = da.concatenate(raw_train_outputs, axis=0)

In [10]:
pt_ta = PowerTransformer()
pt_pr = PowerTransformer()

def fit_transform_channel(inp, channel_idx):
    x = inp[:, channel_idx, :, :]
    x_2d = x.reshape((x.shape[0], -1))
    
    x_np = x_2d.compute()
    pt = pt_ta if channel_idx == 0 else pt_pr
    x_trans = pt.fit_transform(x_np)
    
    x_trans_da = da.from_array(x_trans, chunks=x.chunksize[:1] + (x_trans.shape[1],))
    x_trans_da = x_trans_da.reshape((x.shape[0], 48, 72))
    return x_trans_da

def transform_channel(inp, channel_idx):
    x = inp[:, channel_idx, :, :]
    x_2d = x.reshape((x.shape[0], -1))
    
    x_np = x_2d.compute()
    pt = pt_ta if channel_idx == 0 else pt_pr
    x_trans = pt.transform(x_np)
    
    x_trans_da = da.from_array(x_trans, chunks=x.chunksize[:1] + (x_trans.shape[1],))
    x_trans_da = x_trans_da.reshape((x.shape[0], 48, 72))
    return x_trans_da

def inverse_channel(inp, channel_idx):
    x = inp[:, channel_idx, :, :]
    x_2d = x.reshape(inp.shape[0], -1) 
    pt = pt_ta if channel_idx == 0 else pt_pr

    x_inv = pt.inverse_transform(x_2d)
    return x_inv.reshape(x.shape[0], 48, 72)

In [11]:
train_channel0 = fit_transform_channel(raw_train_outputs, 0)
train_channel1 = fit_transform_channel(raw_train_outputs, 1)

val_channel0 = transform_channel(raw_val_outputs, 0)
val_channel1 = transform_channel(raw_val_outputs, 1)

raw_train_outputs = da.stack([train_channel0, train_channel1], axis=1)
raw_val_outputs = da.stack([val_channel0, val_channel1], axis=1)

normalizer.set_input_statistics(
    mean=da.nanmean(raw_train_inputs, axis=(0, 2, 3), keepdims=True).compute(),
    std=da.nanstd(raw_train_inputs, axis=(0, 2, 3), keepdims=True).compute(),
)
normalizer.set_output_statistics(
    mean=da.nanmean(raw_train_outputs, axis=(0, 2, 3), keepdims=True).compute(),
    std=da.nanstd(raw_train_outputs, axis=(0, 2, 3), keepdims=True).compute(),
)

/opt/conda/lib/python3.11/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/opt/conda/lib/python3.11/site-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)


In [12]:
train_input, train_output = [], []
val_input, val_output = [], []

def create_sequences(x, y, seq_len, pad_value=0.0):
    X_seq, Y_seq = [], []
    T = len(x)
    C, H, W = x.shape[1:]

    for t in range(T):
        start_idx = max(0, t - seq_len + 1)
        actual_seq = x[start_idx:t+1]

        padded_seq = np.full((seq_len, C, H, W), fill_value=pad_value, dtype=x.dtype)
        padded_seq[-(t - start_idx + 1):] = actual_seq

        X_seq.append(padded_seq)
        Y_seq.append(y[t])

    return X_seq, Y_seq

for ssp in train_ssps:
    x_train = normalizer.normalize(ssp_dc[ssp][0], 'input')

    y_train_channel0 = transform_channel(ssp_dc[ssp][1], 0)
    y_train_channel1 = transform_channel(ssp_dc[ssp][1], 1)
    y_train = da.stack([y_train_channel0, y_train_channel1], axis=1)
    y_train = normalizer.normalize(y_train, 'output')
    
    if ssp == "ssp370":
        x_val = normalizer.normalize(x[-test_months:], 'input')

        y_val_channel0 = transform_channel(ssp_dc[ssp][1], 0)
        y_val_channel1 = transform_channel(ssp_dc[ssp][1], 1)
        y_val = da.stack([y_val_channel0, y_val_channel1], axis=1)
        y_val = normalizer.normalize(y_val, 'output')

        X_val, Y_val = create_sequences(x_val, y_val, seq_len=12)
        
        val_input.extend(X_val)
        val_output.extend(Y_val)
    
    X_seq, Y_seq = create_sequences(x_train, y_train, seq_len=12)
    train_input.extend(X_seq)
    train_output.extend(Y_seq)

len(train_input)

2703

In [13]:
train_input = da.stack(train_input)
train_output = da.stack(train_output)

val_input = da.stack(val_input)
val_output = da.stack(val_output)

train_input.shape

(2703, 12, 5, 48, 72)

In [14]:
raw_test_input, raw_test_output = load_ssp(test_ssp)
raw_test_input = raw_test_input[-test_months:]
raw_test_output = raw_test_output[-test_months:]
x_test = normalizer.normalize(raw_test_input, "input")
X_test, Y_test = create_sequences(x_test, raw_test_output, seq_len=12)

test_input = da.stack(X_test)
test_output = da.stack(Y_test)

test_input.shape, test_output.shape

((360, 12, 5, 48, 72), (360, 2, 48, 72))

In [15]:
train_dataset = ClimateDataset(train_input, train_output)
val_dataset = ClimateDataset(val_input, val_output)
test_dataset = ClimateDataset(test_input, test_output, output_is_normalized=False)

Creating dataset with 2703 samples...
Creating dataset with 360 samples...
Creating dataset with 360 samples...


In [16]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)

test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)

In [17]:
class CNNEncoder(nn.Module):
    def __init__(self, in_channels, embed_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            #nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            #nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.fc = nn.Linear(256 * 48 * 72, embed_dim)

    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)  

In [18]:
class RNN(nn.Module):
    def __init__(self, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x_seq):
        out, _ = self.rnn(x_seq)
        out = out[:, -1]
        return self.fc(out)   

In [19]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, hidden_dim, output_dim, num_heads=4, num_layers=2, dropout=0):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, output_dim)

    def forward(self, x_seq):
        out = self.transformer(x_seq)
        out = out[:, -1]     
        return self.fc(out)

In [20]:
class SpatiotemporalModel(nn.Module):
    def __init__(self, in_channels, out_channels, embed_dim, hidden_dim, H, W, temporal_model):
        super().__init__()
        self.encoder = CNNEncoder(in_channels, embed_dim)
        self.temporal = temporal_model(embed_dim, hidden_dim, out_channels * H * W)
        
        self.H = H
        self.W = W
        self.out_channels = out_channels
        self.in_channels = in_channels

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_seq = []
        for t in range(T):
            x_t = self.encoder(x[:, t])
            x_seq.append(x_t)
        x_seq = torch.stack(x_seq, dim=1)
        
        out = self.temporal(x_seq)
        
        return out.view(B, self.out_channels, H, W)

In [21]:
model_tas = SpatiotemporalModel(in_channels=5, out_channels=1, embed_dim=32, hidden_dim=64, H=48, W=72, temporal_model=RNN).to(device)
model_pr  = SpatiotemporalModel(in_channels=5, out_channels=1, embed_dim=32, hidden_dim=64, H=48, W=72, temporal_model=RNN).to(device)

optimizer_tas = optim.Adam(model_tas.parameters(), lr=0.001)
optimizer_pr  = optim.Adam(model_pr.parameters(), lr=0.001)

criterion = nn.MSELoss()

for epoch in range(15):
    model_tas.train()
    model_pr.train()
    total_train_loss_tas = 0
    total_train_loss_pr = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb_tas = yb[:, 0:1, :, :].to(device)
        yb_pr = yb[:, 1:2, :, :].to(device)

        # Train model_tas
        pred_tas = model_tas(xb)
        loss_tas = criterion(pred_tas, yb_tas)
        optimizer_tas.zero_grad()
        loss_tas.backward()
        optimizer_tas.step()
        total_train_loss_tas += loss_tas.item()

        # Train model_pr
        pred_pr = model_pr(xb)
        loss_pr = criterion(pred_pr, yb_pr)
        optimizer_pr.zero_grad()
        loss_pr.backward()
        optimizer_pr.step()
        total_train_loss_pr += loss_pr.item()

    model_tas.eval()
    model_pr.eval()

    with torch.no_grad():
        all_preds_tas, all_targets_tas = [], []
        all_preds_pr, all_targets_pr = [], []
        total_val_loss_tas = 0
        total_val_loss_pr = 0

        for xb, yb in val_loader:
            xb = xb.to(device)
            yb_tas = yb[:, 0:1, :, :].to(device)
            yb_pr = yb[:, 1:2, :, :].to(device)

            # model_tas
            pred_tas = model_tas(xb)
            loss_tas = criterion(pred_tas, yb_tas)
            total_val_loss_tas += loss_tas.item()
            all_preds_tas.append(pred_tas.cpu())
            all_targets_tas.append(yb_tas.cpu())

            # model_pr
            pred_pr = model_pr(xb)
            loss_pr = criterion(pred_pr, yb_pr)
            total_val_loss_pr += loss_pr.item()
            all_preds_pr.append(pred_pr.cpu())
            all_targets_pr.append(yb_pr.cpu())

        # Inverse transform + metrics for tas
        p_tas = torch.cat(all_preds_tas, dim=0).numpy()
        t_tas = torch.cat(all_targets_tas, dim=0).numpy()
        p_tas = inverse_channel(normalizer.inverse_transform_output_1dim(p_tas, dim=0), 0)
        t_tas = inverse_channel(normalizer.inverse_transform_output_1dim(t_tas, dim=0), 0)

        # p_tas = p_tas.squeeze(1)
        # t_tas = t_tas.squeeze(1)
        time = np.arange(p_tas.shape[0])

        p_xr = xr.DataArray(p_tas, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
        t_xr = xr.DataArray(t_tas, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})

        rmse = np.sqrt(((p_xr - t_xr) ** 2).weighted(area_weights).mean(("time", "y", "x")).item())
        mean_rmse = np.sqrt(((p_xr.mean("time") - t_xr.mean("time")) ** 2).weighted(area_weights).mean(("y", "x")).item())
        std_mae = np.abs(p_xr.std("time") - t_xr.std("time")).weighted(area_weights).mean(("y", "x")).item()
        print(f"[tas] Epoch {epoch+1}: Train Loss = {total_train_loss_tas / len(train_loader):.4f}, Valid Loss = {total_val_loss_tas / len(val_loader):.4f}")
        print(f"[tas] RMSE={rmse:.4f}, Time-Mean RMSE={mean_rmse:.4f}, Time-Stddev MAE={std_mae:.4f}")

        # Inverse transform + metrics for pr
        p_pr = torch.cat(all_preds_pr, dim=0).numpy()
        t_pr = torch.cat(all_targets_pr, dim=0).numpy()
        p_pr = inverse_channel(normalizer.inverse_transform_output_1dim(p_pr, dim=1), 1)
        t_pr = inverse_channel(normalizer.inverse_transform_output_1dim(t_pr, dim=1), 1)

        # p_pr = p_pr.squeeze(1)
        # t_pr = t_pr.squeeze(1)

        p_xr = xr.DataArray(p_pr, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
        t_xr = xr.DataArray(t_pr, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})

        rmse = np.sqrt(((p_xr - t_xr) ** 2).weighted(area_weights).mean(("time", "y", "x")).item())
        mean_rmse = np.sqrt(((p_xr.mean("time") - t_xr.mean("time")) ** 2).weighted(area_weights).mean(("y", "x")).item())
        std_mae = np.abs(p_xr.std("time") - t_xr.std("time")).weighted(area_weights).mean(("y", "x")).item()
        print(f"[pr] Epoch {epoch+1}: Train Loss = {total_train_loss_pr / len(train_loader):.4f}, Valid Loss = {total_val_loss_pr / len(val_loader):.4f}")
        print(f"[pr] RMSE={rmse:.4f}, Time-Mean RMSE={mean_rmse:.4f}, Time-Stddev MAE={std_mae:.4f}")

    print('\n')

[tas] Epoch 1: Train Loss = 0.7604, Valid Loss = 1.4145
[tas] RMSE=5.5436, Time-Mean RMSE=2.2672, Time-Stddev MAE=3.3100


IndexError: index 1 is out of bounds for axis 1 with size 1

In [ ]:
def plot_comparison(true_xr, pred_xr, title, cmap='viridis', diff_cmap='RdBu_r', metric=None):
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))

    vmin = min(true_xr.min().item(), pred_xr.min().item())
    vmax = max(true_xr.max().item(), pred_xr.max().item())

    # Ground truth
    true_xr.plot(ax=axs[0], cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=True)
    axs[0].set_title(f"{title} (Ground Truth)")

    # Prediction
    pred_xr.plot(ax=axs[1], cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=True)
    axs[1].set_title(f"{title} (Prediction)")

    # Difference
    diff = pred_xr - true_xr
    abs_max = np.max(np.abs(diff))
    diff.plot(ax=axs[2], cmap=diff_cmap, vmin=-abs_max, vmax=abs_max, add_colorbar=True)
    axs[2].set_title(f"{title} (Difference) {f'- {metric:.4f}' if metric else ''}")

    plt.tight_layout()
    plt.show()

In [ ]:
lat = spatial_template.y.values
lon = spatial_template.x.values
time = np.arange(all_preds.shape[0])

for i, var in enumerate(output_vars):
    pred_xr = xr.DataArray(all_preds[:, i], dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
    true_xr = xr.DataArray(all_targets[:, i], dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})

    # --- Time Mean ---
    plot_comparison(true_xr.mean("time"), pred_xr.mean("time"), f"{var} Val Time-Mean")

    # --- Time Stddev ---
    plot_comparison(true_xr.std("time"), pred_xr.std("time"), f"{var} Val Time-Stddev", cmap="plasma")

    # --- Random timestep ---
    t_idx = np.random.randint(0, len(time))
    plot_comparison(true_xr.isel(time=t_idx), pred_xr.isel(time=t_idx), f"{var} Val Sample Timestep {t_idx}")

In [ ]:
model.eval()

test_predictions = []
test_targets = []

with torch.no_grad():
    for xb, yb in test_dataloader:
        xb, yb = xb.to(device), yb.to(device)

        pred = model(xb)
        loss = criterion(pred, yb)

        test_predictions.append(pred.cpu())
        test_targets.append(yb.cpu())

test_predictions = torch.cat(test_predictions, dim=0)
test_targets = torch.cat(test_targets, dim=0)

test_predictions = normalizer.inverse_transform_output(test_predictions.detach().cpu().numpy())
test_targets = normalizer.inverse_transform_output(test_targets.detach().cpu().numpy())

In [ ]:
lat = spatial_template.y.values
lon = spatial_template.x.values
time = np.arange(test_predictions.shape[0])

rows = []
for t_idx, t in enumerate(time):
    for var_idx, var in enumerate(output_vars):
        for y_idx, y in enumerate(lat):
            for x_idx, x in enumerate(lon):
                row_id = f"t{t_idx:03d}_{var}_{y:.2f}_{x:.2f}"
                pred = test_predictions[t_idx, var_idx, y_idx, x_idx]
                rows.append({"ID": row_id, "Prediction": pred})

df = pd.DataFrame(rows)
os.makedirs("submissions", exist_ok=True)
filepath = f"submissions/kaggle_submission_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filepath, index=False)
print(f"✅ Submission saved to: {filepath}")